In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.utils import resample
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, classification_report

In [3]:
train_df = pd.read_csv("data/train.tsv", sep="\t", header=None)
train_df.columns = ["text", "label", "id"]

print(train_df.head())
print(train_df.shape)

                                                text label       id
0  My favourite food is anything I didn't have to...    27  eebbqej
1  Now if he does off himself, everyone will thin...    27  ed00q6i
2                     WHY THE FUCK IS BAYLESS ISOING     2  eezlygj
3                        To make her feel threatened    14  ed7ypvh
4                             Dirty Southern Wankers     3  ed0bdzj
(43410, 3)


In [4]:
train_df["label"] = train_df["label"].astype(str).apply(lambda x: x.split(",")[0])
train_df["label"] = train_df["label"].astype(int)

In [5]:
def map_emotion(label):
    label = int(label)

    if label in [2, 3, 10, 25]:
        return 0  # sadness
    elif label in [1, 7, 22, 26]:
        return 1  # anger
    elif label in [15, 18, 27]:
        return 2  # joy
    elif label in [4, 6, 9]:
        return 3  # fear
    elif label in [14, 20, 17]:
        return 4  # love
    else:
        return 5  # neutral/others

train_df["label"] = train_df["label"].apply(map_emotion)

In [6]:
df_list = []

for label in train_df["label"].unique():
    temp = train_df[train_df["label"] == label]

    # reduce huge classes, expand small ones
    if len(temp) > 3000:
        temp = temp.sample(3000, random_state=42)
    else:
        temp = resample(temp, replace=True, n_samples=3000, random_state=42)

    df_list.append(temp)

train_df = pd.concat(df_list)

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    train_df["text"],
    train_df["label"],
    test_size=0.2,
    random_state=42,
    stratify=train_df["label"]
)

In [8]:
vectorizer = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1,2),
    stop_words="english",
    min_df=2
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [9]:
model = SGDClassifier(
    loss="log_loss",
    max_iter=2000,
    tol=1e-3,
    random_state=42
)

model.fit(X_train_vec, y_train)

,loss,'log_loss'
,penalty,'l2'
,alpha,0.0001
,l1_ratio,0.15
,fit_intercept,True
,max_iter,2000
,tol,0.001
,shuffle,True
,verbose,0
,epsilon,0.1
,n_jobs,None


In [10]:
pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, pred))
print("\nClassification Report:\n")
print(classification_report(y_test, pred))

Accuracy: 0.5144444444444445

Classification Report:

              precision    recall  f1-score   support

           0       0.49      0.54      0.52       600
           1       0.59      0.52      0.55       600
           2       0.38      0.43      0.41       600
           3       0.41      0.39      0.40       600
           4       0.71      0.70      0.71       600
           5       0.53      0.50      0.52       600

    accuracy                           0.51      3600
   macro avg       0.52      0.51      0.52      3600
weighted avg       0.52      0.51      0.52      3600



In [11]:
from transformers import pipeline

classifier = pipeline("text-classification",
                      model="j-hartmann/emotion-english-distilroberta-base")

print(classifier("I am very happy today"))
print(classifier("I feel very angry and frustrated"))

config.json: 0.00B [00:00, ?B/s]

C:\Users\vsaur\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\vsaur\.cache\huggingface\hub\models--j-hartmann--emotion-english-distilroberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


pytorch_model.bin:   0%|          | 0.00/329M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/294 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/329M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

[{'label': 'joy', 'score': 0.9774639010429382}]
[{'label': 'anger', 'score': 0.994340181350708}]


In [ ]:
import pickle

# Save trained ML model
with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

# Save TF-IDF vectorizer
with open("vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

print("Model and vectorizer saved successfully")

Model and vectorizer saved successfully!
